In [19]:
import os


def escape_string(text):
    result = []
    for c in text:
        if c == '"':
            result.append('\\"')
        elif c == "\\":
            result.append("\\\\")
        elif c == "\n":
            result.append("\\n")
        elif c == "\r":
            result.append("\\r")
        elif c == "\t":
            result.append("\\t")
        else:
            result.append(c)
    return "".join(result)


def create_db(paths, output_json_path):
    parallelizable_cnt = 0

    with open(output_json_path, "w") as f:
        for path in paths:
            parallelizable = False
            with open(os.path.join(path, "code.c"), "r") as code_f:
                code_content = code_f.read()

            if os.path.exists(os.path.join(path, "pragma.c")):
                parallelizable = True
                parallelizable_cnt += 1
                with open(os.path.join(path, "pragma.c"), "r") as pragma_f:
                    pragma = pragma_f.read()
            else:
                pragma = ""

            json_entry = (
                "{"
                f'"code": "{escape_string(code_content)}", '
                f'"label": "{parallelizable}", '
                f'"pragma": "{escape_string(pragma)}"'
                "}\n"
            )
            f.write(json_entry)

    return {
        "parallelizable": parallelizable_cnt,
        "non_parallelizable": len(paths) - parallelizable_cnt,
        "total": len(paths),
    }

In [20]:
paths = [
    os.path.join(
        "/home/mohamed-ashraf/Desktop/projects/accelera/DB_TEST/database/",
        directory,
    )
    for directory in os.listdir(
        "/home/mohamed-ashraf/Desktop/projects/accelera/DB_TEST/database/"
    )
]

output_json_path = "test.json"

output = create_db(paths, output_json_path)
print(output)

{'parallelizable': 154, 'non_parallelizable': 319, 'total': 473}


In [ ]:
import json

import numpy as np
import pandas as pd

from accelera.src.utils.parallelizer import extract_features
from accelera.src.utils.parallelizer import pragma_to_class
from accelera.src.utils.parallelizer import vectorize_features

jsonl_path = "data.json"
vectors = []
labels = []

with open(jsonl_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        code = obj["code"]
        pragma = obj["pragma"]
        features = extract_features(code)
        vectors.append(vectorize_features(features))
        labels.append(pragma_to_class(obj["label"], pragma))

vectors_array = np.array(vectors)
df = pd.DataFrame(
    vectors_array,
    columns=[f"feature_{i}" for i in range(vectors_array.shape[1])],
)
df["label"] = labels
df.to_csv("data.csv", index=False)
print(f"Saved {len(df)} rows to data.csv")

Setting up accelera module path...
Saved 32595 rows to data.csv


In [2]:
data = pd.read_csv("data.csv")

feature_cols = [col for col in data.columns if col.startswith("feature_")]
vectors = data[feature_cols].values
labels = data["label"].values

print(f"Vector type: {type(vectors[0])}")
print(f"Vector shape: {vectors[0].shape}")
print(f"First vector:\n{vectors[0]}")

Vector type: <class 'numpy.ndarray'>
Vector shape: (40,)
First vector:
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         1.         0.
 0.         0.         0.         0.         1.         0.2
 0.         0.25       0.25       0.35643566 0.7128713  0.06930693
 0.42574257 0.7821782  0.         0.         0.         0.
 0.         0.         0.         0.         0.08       0.04
 0.4        0.05       0.05       0.        ]
